# Lab 9: Named Entity Recognition (NER)

**Name:** Mujtaba Alnuwaysir
**ID:** 2240007941

In [ ]:
# Import the spaCy library and load the English language model
import spacy

nlp = spacy.load("en_core_web_sm")

# Sample text for NER
text = "Apple Inc. was founded by Steve Jobs in Cupertino, California. It is known for its iPhone and Mac products."
doc = nlp(text)

print(f"Entities recognized in text: {doc.ents}")

# Extract and print named entities
for entity in doc.ents:
    print(f"Entity: {entity.text}, Label: {entity.label_}")

Entities recognized in text: (Apple Inc., Steve Jobs, Cupertino, California, Mac)
Entity: Apple Inc., Label: ORG
Entity: Steve Jobs, Label: PERSON
Entity: Cupertino, Label: GPE
Entity: California, Label: GPE
Entity: Mac, Label: ORG


In [ ]:
# Iterate through recognized entities and print their details
for ent in doc.ents:
    print(f"Entity: {ent.text}")
    print(f"Label: {ent.label_}")
    print(f"Start Position: {ent.start}")
    print(f"End Position: {ent.end}")
    print(f"Entity Type: {spacy.explain(ent.label_)}")
    print("")

Entity: Apple Inc.
Label: ORG
Start Position: 0
End Position: 2
Entity Type: Companies, agencies, institutions, etc.

Entity: Steve Jobs
Label: PERSON
Start Position: 5
End Position: 7
Entity Type: People, including fictional

Entity: Cupertino
Label: GPE
Start Position: 8
End Position: 9
Entity Type: Countries, cities, states

Entity: California
Label: GPE
Start Position: 10
End Position: 11
Entity Type: Countries, cities, states

Entity: Mac
Label: ORG
Start Position: 19
End Position: 20
Entity Type: Companies, agencies, institutions, etc.



In [ ]:
from spacy.training.iob_utils import offsets_to_biluo_tags

# The text to annotate
text = "Apple Inc. was founded by Steve Jobs in Cupertino, California. It is known for its iPhone and Mac products."

# Add custom entities annotations manually
annotations = [
    (text.find("Apple Inc."), text.find("Apple Inc.") + len("Apple Inc."), "ORG"),
    (text.find("Steve Jobs"), text.find("Steve Jobs") + len("Steve Jobs"), "PERSON"),
    (text.find("Cupertino, California"), text.find("Cupertino, California") + len("Cupertino, California"), "GPE")
]

# Check entity alignment
doc = nlp.make_doc(text)
tags = offsets_to_biluo_tags(doc, annotations)
print("Alignment Tags:", tags[:10])

Alignment Tags: ['B-ORG', 'L-ORG', 'O', 'O', 'O', 'B-PERSON', 'L-PERSON', 'O', 'B-GPE', 'I-GPE']


In [ ]:
import spacy
from spacy.training import Example

# Define the training text
text = "Microsoft was founded by Bill Gates and Paul Allen in Albuquerque, New Mexico. It is known for its Windows operating system and Office software products."

# Define the entity annotations
entities = [
    (text.find("Microsoft"), text.find("Microsoft") + len("Microsoft"), "ORG"),
    (text.find("Bill Gates"), text.find("Bill Gates") + len("Bill Gates"), "PERSON"),
    (text.find("Paul Allen"), text.find("Paul Allen") + len("Paul Allen"), "PERSON"),
    (text.find("Albuquerque, New Mexico"), text.find("Albuquerque, New Mexico") + len("Albuquerque, New Mexico"), "GPE"),
    (text.find("Windows"), text.find("Windows") + len("Windows"), "PRODUCT"),
    (text.find("Office"), text.find("Office") + len("Office"), "PRODUCT")
]

# Create a blank English model
nlp = spacy.blank("en")

# Add the NER pipeline component
ner = nlp.add_pipe("ner")

# Add entity labels
for _, _, label in entities:
    ner.add_label(label)

# Create the training example
doc = nlp.make_doc(text)
example = Example.from_dict(doc, {"entities": entities})
examples = [example]

# Initialize the model
nlp.initialize(lambda: examples)

# Train the model for 20 iterations
for iteration in range(20):
    losses = {}
    nlp.update(examples, losses=losses)
    print(f"Iteration {iteration + 1}: {losses}")

# Save the trained model
nlp.to_disk("custom_ner_model")
print("Model saved successfully.")

Iteration 1: {'ner': 15.944444119930267}
Iteration 2: {'ner': 26.4891037940979}
Iteration 3: {'ner': 26.236817955970764}
Iteration 4: {'ner': 25.842512607574463}
Iteration 5: {'ner': 25.263886332511902}
Iteration 6: {'ner': 24.363179743289948}
Iteration 7: {'ner': 23.159571528434753}
Iteration 8: {'ner': 21.223674297332764}
Iteration 9: {'ner': 18.67949467897415}
Iteration 10: {'ner': 15.011636018753052}
Iteration 11: {'ner': 10.25610064715147}
Iteration 12: {'ner': 7.344854572787881}
Iteration 13: {'ner': 5.563017713371664}
Iteration 14: {'ner': 6.553363510145573}
Iteration 15: {'ner': 7.77471859139041}
Iteration 16: {'ner': 6.523938398860992}
Iteration 17: {'ner': 4.706881319722015}
Iteration 18: {'ner': 5.015430062770065}
Iteration 19: {'ner': 3.9450132116512577}
Iteration 20: {'ner': 2.945047710281301}
Model saved successfully.


In [ ]:
# Load and use the custom model for entity recognition
custom_nlp = spacy.load("custom_ner_model")
doc2 = custom_nlp(text)

# Entity visualization
from spacy import displacy

displacy.render(doc2, style="ent", jupyter=True)

In [ ]:
# Ask the user to enter a sentence
user_text = input("Enter a sentence: ")

# Identify named entities
doc = nlp(user_text)

# Print the named entities
print("Named Entities:")
for entity in doc.ents:
    print(f"Entity: {entity.text}, Label: {entity.label_}")

# Print the details of each entity
print("\nEntity Details:")
for ent in doc.ents:
    print(f"Entity: {ent.text}")
    print(f"Label: {ent.label_}")
    print(f"Start Position: {ent.start}")
    print(f"End Position: {ent.end}")
    print(f"Entity Type: {spacy.explain(ent.label_)}")
    print("")

Named Entities:

Entity Details:


In [ ]:
# Define the paragraph
paragraph = (
    "This agreement is made and entered into by and between "
    "'Abc & Co.' and 'Bcd LLC' for term 1 year starting from "
    "April 1, 2020, hereinafter collectively referred to as the Parties. "
    "Samuel P. Jackson in the place (New York) and on the date written below, "
    "with the following terms and conditions."
)

# Add entities that may not be identified by the existing model
ruler = nlp.add_pipe("entity_ruler", before="ner")
patterns = [
    {"label": "ORG", "pattern": "Abc & Co."},
    {"label": "ORG", "pattern": "Bcd LLC"},
    {"label": "PERSON", "pattern": "Samuel P. Jackson"}
]
ruler.add_patterns(patterns)

# Identify named entities
doc = nlp(paragraph)

# Print all entities
print("Named Entities:")
for ent in doc.ents:
    print(f"Entity: {ent.text}, Label: {ent.label_}")

# Count PERSON and ORG entities
person_count = sum(1 for ent in doc.ents if ent.label_ == "PERSON")
org_count = sum(1 for ent in doc.ents if ent.label_ == "ORG")

print("\nEntity Counts:")
print(f"PERSON: {person_count}")
print(f"ORG: {org_count}")

# Visualize the entities
from spacy import displacy

displacy.render(doc, style="ent", jupyter=True)

Named Entities:
Entity: Abc & Co., Label: ORG
Entity: Bcd LLC, Label: ORG
Entity: Samuel P. Jackson, Label: PERSON

Entity Counts:
PERSON: 1
ORG: 2


In [ ]:
import spacy
from spacy.training import Example

# Define the training text
text = "Microsoft was founded by Bill Gates and Paul Allen in Albuquerque, New Mexico. It is known for its Windows operating system and Office software products."

# Define entity annotations
entities = [
    (0, 9, "ORG"),
    (25, 35, "PERSON"),
    (40, 50, "PERSON"),
    (54, 77, "GPE"),
    (104, 111, "PRODUCT"),
    (132, 138, "PRODUCT")
]

# Create a blank English NLP model
nlp = spacy.blank("en")

# Add the NER component
ner = nlp.add_pipe("ner")

# Add entity labels
for _, _, label in entities:
    ner.add_label(label)

# Create a training example using Example.from_dict()
doc = nlp.make_doc(text)
example = Example.from_dict(doc, {"entities": entities})
examples = [example]

# Initialize the model
nlp.initialize(lambda: examples)

# Train the NER model for 20 iterations
for iteration in range(20):
    losses = {}
    nlp.update(examples, losses=losses)
    print(f"Iteration {iteration + 1}: {losses}")

# Save the trained model
nlp.to_disk("custom_ner_model")
print("Model saved successfully.")

d:\DONLWD\New folder\Lib\site-packages\spacy\training\iob_utils.py:149: UserWarning: [W030] Some entities could not be aligned in the text "Microsoft was founded by Bill Gates and Paul Allen..." with entities "[(0, 9, 'ORG'), (25, 35, 'PERSON'), (40, 50, 'PERS...". Use `spacy.training.offsets_to_biluo_tags(nlp.make_doc(text), entities)` to check the alignment. Misaligned entities ('-') will be ignored during training.
  warnings.warn(


Iteration 1: {'ner': 14.944444000720978}
Iteration 2: {'ner': 22.91193312034011}
Iteration 3: {'ner': 22.670880008488894}
Iteration 4: {'ner': 22.26831318065524}
Iteration 5: {'ner': 21.599012948572636}
Iteration 6: {'ner': 20.43075855821371}
Iteration 7: {'ner': 18.6937866024673}
Iteration 8: {'ner': 15.89389831200242}
Iteration 9: {'ner': 11.869560707360506}
Iteration 10: {'ner': 7.836582779884338}
Iteration 11: {'ner': 4.949106765910983}
Iteration 12: {'ner': 3.982609580969438}
Iteration 13: {'ner': 3.870700418803608}
Iteration 14: {'ner': 3.2598315127397655}
Iteration 15: {'ner': 5.760428741985379}
Iteration 16: {'ner': 5.0695792544591995}
Iteration 17: {'ner': 4.612169886577703}
Iteration 18: {'ner': 30.559909795600106}
Iteration 19: {'ner': 6.513933047693726}
Iteration 20: {'ner': 2.6628193159551756}
Model saved successfully.


In [ ]:
# Load the saved custom NER model
custom_nlp = spacy.load("custom_ner_model")

# Identify named entities in a sentence
test_text = "Microsoft was founded by Bill Gates in Albuquerque, New Mexico."
doc = custom_nlp(test_text)

# Print the recognized entities
print("Named Entities:")
for ent in doc.ents:
    print(f"Entity: {ent.text}, Label: {ent.label_}")

# Visualize the recognized entities
from spacy import displacy

displacy.render(doc, style="ent", jupyter=True)

Named Entities:
Entity: Microsoft, Label: ORG
Entity: Bill Gates, Label: PERSON
Entity: Albuquerque, New Mexico., Label: GPE
